In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [1]:
import os 
import re 
import numpy as np 
import pandas as pd 
from collections import Counter 
import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader 
from sklearn.model_selection import train_test_split

**Dataset**

In [2]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample_submission=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [18]:
train.shape,test.shape

((2000, 8), (500, 7))

In [16]:
train.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [17]:
test.head(2)

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."


In [19]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

**Model-1**:
Attention-based BiLSTM Ranker trained from scratch using BCE loss.

**config**

In [4]:
# MAX_LEN = 256 
# EMBED_DIM = 256 
# HIDDEN_DIM = 256 
# BATCH_SIZE = 64 
# EPOCHS = 10 
# LR = 1e-3 
# LABELS = ["A", "B", "C", "D", "E"]

In [5]:
# #Tokenizer

# def clean_text(text):
#     text = str(text).lower()
#     text = re.sub(r"[^a-z0-9 ]", " ", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

# def tokenize(text):
#     return clean_text(text).split()


In [6]:
# #Vocab

# def build_vocab(df):

#     counter = Counter()

#     for _, row in df.iterrows():

#         counter.update(tokenize(row["prompt"]))

#         for col in LABELS:
#             counter.update(tokenize(row[col]))

#     vocab = {
#         "<PAD>": 0,
#         "<UNK>": 1
#     }

#     for word, freq in counter.items():
#         if freq >= 2:
#             vocab[word] = len(vocab)

#     return vocab

# def encode(text, vocab):

#     tokens = tokenize(text)

#     ids = [vocab.get(t, 1) for t in tokens]

#     ids = ids[:MAX_LEN]

#     if len(ids) < MAX_LEN:
#         ids += [0] * (MAX_LEN - len(ids))

#     return ids


In [8]:
# #Dataset

# class MCQDataset(Dataset):

#     def __init__(self, df, vocab, train_mode=True):

#         self.samples = []

#         for _, row in df.iterrows():

#             prompt = str(row["prompt"])

#             if train_mode:

#                 answer = row["answer"]

#                 for label in LABELS:

#                     option = str(row[label])

#                     text = prompt + " [SEP] " + option

#                     target = 1 if label == answer else 0

#                     self.samples.append(
#                         (
#                             encode(text, vocab),
#                             target
#                         )
#                     )

#             else:

#                 question_id = row["id"]

#                 for label in LABELS:

#                     option = str(row[label])

#                     text = prompt + " [SEP] " + option

#                     self.samples.append(
#                         (
#                             question_id,
#                             label,
#                             encode(text, vocab)
#                         )
#                     )

#         self.train_mode = train_mode

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):

#         if self.train_mode:

#             x, y = self.samples[idx]

#             return (
#                 torch.tensor(x, dtype=torch.long),
#                 torch.tensor(y, dtype=torch.float)
#             )

#         qid, label, x = self.samples[idx]

#         return (
#             qid,
#             label,
#             torch.tensor(x, dtype=torch.long)
#         )

In [9]:
# # Attention

# class AttentionPool(nn.Module):

#     def __init__(self, hidden_size):

#         super().__init__()

#         self.attn = nn.Linear(hidden_size, 1)

#     def forward(self, x):

#         scores = self.attn(x)

#         weights = torch.softmax(scores, dim=1)

#         pooled = (weights * x).sum(dim=1)

#         return pooled

In [10]:
# # Model

# class MCQRanker(nn.Module):

#     def __init__(self, vocab_size):

#         super().__init__()

#         self.embedding = nn.Embedding(
#             vocab_size,
#             EMBED_DIM,
#             padding_idx=0
#         )

#         self.lstm = nn.LSTM(
#             EMBED_DIM,
#             HIDDEN_DIM,
#             batch_first=True,
#             bidirectional=True
#         )

#         self.pool = AttentionPool(HIDDEN_DIM * 2)

#         self.classifier = nn.Sequential(
#             nn.Linear(HIDDEN_DIM * 2, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 1)
#         )

#     def forward(self, x):

#         emb = self.embedding(x)

#         out, _ = self.lstm(emb)

#         pooled = self.pool(out)

#         logits = self.classifier(pooled)

#         return logits.squeeze(-1)

In [11]:
# #Train...................
# def train_epoch(model, loader, optimizer, criterion):

#     model.train()

#     total_loss = 0

#     for x, y in loader:

#         x = x.to(DEVICE)
#         y = y.to(DEVICE)

#         optimizer.zero_grad()

#         logits = model(x)

#         loss = criterion(logits, y)

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     return total_loss / len(loader)

In [12]:
# #Validation
# @torch.no_grad()
# def evaluate(model, loader, criterion):

#     model.eval()

#     total_loss = 0

#     for x, y in loader:

#         x = x.to(DEVICE)
#         y = y.to(DEVICE)

#         logits = model(x)

#         loss = criterion(logits, y)

#         total_loss += loss.item()

#     return total_loss / len(loader)


In [13]:
# #Training the model................
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# vocab = build_vocab(train_df)

# train_part, valid_part = train_test_split(
#     train_df,
#     test_size=0.1,
#     random_state=42,
#     stratify=train_df["answer"]
# )

# train_ds = MCQDataset(train_part, vocab, True)
# valid_ds = MCQDataset(valid_part, vocab, True)

# train_loader = DataLoader(
#     train_ds,
#     batch_size=BATCH_SIZE,
#     shuffle=True
# )

# valid_loader = DataLoader(
#     valid_ds,
#     batch_size=BATCH_SIZE
# )

# model = MCQRanker(len(vocab)).to(DEVICE)

# criterion = nn.BCEWithLogitsLoss()

# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr=LR
# )

# best_loss = 999

# for epoch in range(EPOCHS):

#     train_loss = train_epoch(
#         model,
#         train_loader,
#         optimizer,
#         criterion
#     )

#     valid_loss = evaluate(
#         model,
#         valid_loader,
#         criterion
#     )

#     print(
#         f"Epoch {epoch+1}/{EPOCHS} "
#         f"Train_loss={train_loss:.4f} "
#         f"Validation_loss={valid_loss:.4f}"
#     )

#     if valid_loss < best_loss:

#         best_loss = valid_loss

#         torch.save(
#             {
#                 "model": model.state_dict(),
#                 "vocab": vocab
#             },
#             "best_model.pt"
#         )

Epoch 1/10 Train_loss=0.4976 Validation_loss=0.3822
Epoch 2/10 Train_loss=0.3029 Validation_loss=0.1684
Epoch 3/10 Train_loss=0.1361 Validation_loss=0.0763
Epoch 4/10 Train_loss=0.0713 Validation_loss=0.0851
Epoch 5/10 Train_loss=0.0489 Validation_loss=0.0118
Epoch 6/10 Train_loss=0.0259 Validation_loss=0.0041
Epoch 7/10 Train_loss=0.0233 Validation_loss=0.0098
Epoch 8/10 Train_loss=0.0146 Validation_loss=0.0015
Epoch 9/10 Train_loss=0.0077 Validation_loss=0.0018
Epoch 10/10 Train_loss=0.0031 Validation_loss=0.0002


In [14]:
# #Inference
# checkpoint = torch.load(
#     "best_model.pt",
#     map_location=DEVICE
# )

# model.load_state_dict(checkpoint["model"])
# model.eval()

# test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# test_ds = MCQDataset(
#     test_df,
#     vocab,
#     train_mode=False
# )

# test_loader = DataLoader(
#     test_ds,
#     batch_size=256,
#     shuffle=False
# )

# # Store scores for each question
# scores = {}

# with torch.no_grad():

#     for qids, labels, x in test_loader:

#         x = x.to(DEVICE)

#         logits = model(x)

#         probs = torch.sigmoid(logits).cpu().numpy()

#         for qid, label, p in zip(qids, labels, probs):

#             # Convert ID to string for consistency
#             qid = str(qid)

#             if qid not in scores:
#                 scores[qid] = {}

#             scores[qid][label] = float(p)
# predictions = []

# for qid in test_df["id"]:

#     qid = str(qid)

#     # Safety fallback
#     if qid not in scores:

#         predictions.append("A B C")
#         continue

#     ranked = sorted(
#         scores[qid].items(),
#         key=lambda x: x[1],
#         reverse=True
#     )

#     top3 = [label for label, score in ranked[:3]]

#     # If somehow fewer than 3 labels exist
#     while len(top3) < 3:
#         for lbl in ["A", "B", "C", "D", "E"]:
#             if lbl not in top3:
#                 top3.append(lbl)
#             if len(top3) == 3:
#                 break

#     predictions.append(" ".join(top3))

**Submission**

In [15]:
# submission = pd.DataFrame(
#     {
#         "ID": test_df["id"],
#         "Prediction": predictions
#     }
# )

# submission.to_csv(
#     "submission.csv",
#     index=False
# )

# print(submission.head())
# print("submission.csv is created successfully")

   ID Prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C
submission.csv is created successfully


**Model-2**